# The ICOS Curve story — figures from real ICOS atmosphere data

Re-creation of the *ICOS Curve story* illustration figures, with the two **synthetic**
station time series replaced by **real ICOS atmosphere CO₂ observations** pulled from
the ICOS Obspack zarr archive over the data-passport proxy at `https://zarr.icos-cp.eu`:

| Used in figures | Station | Obspack group | Years |
|---|---|---|---|
| Fig 1, Fig 2, Fig 3 | **Cabauw, 27 m intake** (`CBW27`) — NL lowland tall tower | `co2` (combined view) | 2021–2025 |
| Fig 1\*, Fig 2, Fig 3 | **Jungfraujoch** (`JFJ0`) — CH alpine, 3580 m | `co2` (combined view) | 2021–2025 |
| Fig 2 | **Izaña** (`IZO0`) — ES subtropical free troposphere, ~2400 m | `co2` (combined view) | 2021–2025 |
| Fig 2 | **Station Nord** (`SNO85`) — DK / Greenland, 81.4 °N high-Arctic | `co2` (combined view) | 2021–2025 |

\* Jungfraujoch is the "mountain background" curve in Fig 1's power-spectrum line and Fig 3's
two-station comparison; the three-time-scales panels of Fig 1 show Cabauw 27 m.

Data are hourly dry-air CO₂ mole fractions (ppm, WMO X2019 scale), kept where the ICOS ATC
manual-QC flag is `U` or `O` (data correct), then reindexed onto a gap-free hourly grid
(missing hours → `NaN`; for the Fourier spectrum the gaps are interpolated / climatology-filled).

Figures written to `./figures/`:

- `fig1_three_scales.png` — the CO₂ signal at three time scales (Cabauw 27 m)
- `fig2_power_spectrum.png` — Fourier power spectrum: Cabauw 27 m, Jungfraujoch, Izaña, Station Nord
- `fig3_sources_sinks.png` — sources & sinks (schematic), two-station comparison, time–space map
  — the original panel **(a)** (a synthetic component decomposition) is omitted, since those
  components don't exist in observed data.

**Run this notebook from the repo root** (`c:\av\python\fluxnet`) so the `datapassport_zarr`
import resolves and `figures/` lands next to it.

Dependencies (already in `requirements.txt`): `xarray`, `zarr<3`, `fsspec`, `aiohttp`,
`numpy`, `pandas`, `matplotlib`. Uncomment the install line in the next cell if needed.

In [ ]:
# !pip install "xarray" "zarr<3" fsspec aiohttp numpy pandas matplotlib

import json
import warnings
import pathlib
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from datapassport_zarr import open_zarr   # repo-local client for the data-passport proxy

warnings.filterwarnings("ignore", message="Mean of empty slice")

plt.rcParams.update({
    "figure.dpi": 140, "savefig.dpi": 160,
    "font.family": "DejaVu Sans", "font.size": 10.5,
    "axes.titlesize": 12, "axes.titleweight": "semibold",
    "axes.labelsize": 10.5,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e8ecf2", "grid.linewidth": 0.7,
    "legend.frameon": False,
})

# Palette — unchanged from the original ICOS Curve story illustrations
C_TREND    = "#1f3a5f"
C_SEASONAL = "#2a9d8f"
C_SYNOPTIC = "#d4a017"
C_DIURNAL  = "#e76f51"
C_ANTHRO   = "#6a4c93"
C_OBS      = "#0b3954"
ICOS_BLUE  = "#0064a4"

# ── Data source ───────────────────────────────────────────────────────────────
STORE_URL = "https://zarr.icos-cp.eu/icos-obspack.zarr"   # data-passport proxy
GROUP     = "co2"                                          # combined-view group (all ATC sites)

# Figures 1 & 3 use these two contrasting stations:
ST_LOW    = "CBW27"       # Cabauw, 27 m intake   -> the "lowland tall tower" series
ST_MT     = "JFJ0"        # Jungfraujoch (alpine, 3580 m) -> the "mountain background" series
# Figure 2 (power spectrum) compares those two plus two clean global-background sites:
ST_IZO    = "IZO0"        # Izaña, Tenerife (subtropical free troposphere, ~2400 m)
ST_NORD   = "SNO85"       # Station Nord / Villum, Greenland (high-Arctic, 81.4 N)
STATIONS  = [ST_LOW, ST_MT, ST_IZO, ST_NORD]

T0, T1    = "2021-01-01", "2025-12-31"
QC_KEEP   = {"U", "O"}    # ICOS ATC manual-QC flags meaning "data correct"

OUTDIR = pathlib.Path("figures")
OUTDIR.mkdir(exist_ok=True)
print("Figures will be written to:", OUTDIR.resolve())

## 1 — Fetch the CO₂ series from the Obspack zarr (via the passport proxy)

The Obspack *combined-view* `co2` group shares one `(station, time_co2)` axis pair across all
ICOS atmosphere sites, so all four stations come out of a single `open_zarr` + `.sel(...)`:

| Used in | Station | Group | Role |
|---|---|---|---|
| Fig 1, Fig 2, Fig 3 | Cabauw, 27 m (`CBW27`) | NL lowland tall tower | the busy "lowland" series |
| Fig 1, Fig 2, Fig 3 | Jungfraujoch (`JFJ0`) | CH alpine, 3580 m | "mountain background" |
| Fig 2 | Izaña (`IZO0`) | ES subtropical free troposphere, ~2400 m | clean low-latitude background |
| Fig 2 | Station Nord (`SNO85`) | DK / Greenland, 81.4 °N | clean high-Arctic background |

Reading the chunks over the proxy mints a **data passport** when the session closes — printed below.

In [ ]:
with open_zarr(STORE_URL, group=GROUP, verbose=True) as src:
    src.record_query({"stations": STATIONS, "time": [T0, T1],
                      "variables": ["co2", "co2_qc_flag"]})
    sub = (
        src._ds[["co2", "co2_qc_flag"]]
            .sel(station=STATIONS)
            .sel(time_co2=slice(T0, T1))
            .compute()
    )
    src.record_query({"surviving_stations": [str(s) for s in sub.station.values]})

passport = src._passport

# Per-station summary — obspack site metadata travels along as coords on `station`
for sid in sub.station.values:
    s     = sub.sel(station=sid)
    flags = sorted({str(f) for f in np.asarray(s["co2_qc_flag"].values)})
    n_tot = int(s["co2"].size)
    n_obs = int(np.isfinite(s["co2"].values).sum())
    print(f"{str(sid):8s}  {str(s['site_name'].values):24s}  "
          f"lat={float(s['lat']):7.3f}  lon={float(s['lon']):7.3f}  "
          f"intake={float(s['intake_height']):4.0f} m  "
          f"obs={n_obs:6d}/{n_tot}  ({100 * n_obs / n_tot:4.1f}%)  "
          f"qc-flags={flags}  "
          f"{str(s['time_co2'].values[0])[:10]}…{str(s['time_co2'].values[-1])[:10]}")

print("\nData-passport minted for this access:")
print(json.dumps(passport, indent=2, default=str))

## 2 — Reindex onto a gap-free hourly grid and fill gaps for the spectrum

The obspack timestamps are snapped to whole hours and every station is laid onto one
gap-free hourly grid spanning exactly **2021-01-01 00:00 … 2025-12-31 23:00** (1826 days ×
24 h = 43824 hourly slots; missing hours stay `NaN`). A second, NaN-free copy of each series
(`*_fft`) is built for the Fourier spectrum: interior gaps ≤ 7 days are time-interpolated,
longer gaps and the ends are filled with the day-of-year climatology.

`co2_low` (Cabauw 27 m) and `co2_mt` (Jungfraujoch) keep the original synthetic script's variable
names so the figure code stays essentially unchanged; `co2_izo` / `co2_nord` are the two extra
background sites used only in the Figure-2 spectrum.

In [ ]:
# Snap obspack timestamps down to the whole hour, then reindex onto a single gap-free hourly
# grid.  The obspack `time_co2` axis is hourly but timestamped at HH:30 (epoch "1972-01-01
# 00:30:00"), so `floor("h")` -> HH:00 with exactly one value per hour (round() would tie-break
# half the timestamps to the previous hour and lose ~half the data to de-duplication).
raw_t = pd.DatetimeIndex(np.asarray(sub["time_co2"].values))
if raw_t.tz is not None:
    raw_t = raw_t.tz_convert("UTC").tz_localize(None)
raw_t = raw_t.floor("h")

idx = pd.date_range(T0, pd.Timestamp(T1) + pd.Timedelta(hours=23), freq="h")   # 1826 days x 24 h

frame = pd.DataFrame(index=raw_t)
for sid in sub.station.values:
    s    = sub.sel(station=sid)
    vals = np.asarray(s["co2"].values, dtype="float64")
    qc   = np.asarray(s["co2_qc_flag"].values, dtype=object)
    keep = np.array([str(f) in QC_KEEP for f in qc])
    frame[str(sid)] = np.where(keep, vals, np.nan)
frame = frame[~frame.index.duplicated(keep="first")].reindex(idx)

co2_low  = frame[ST_LOW ].to_numpy()   # Cabauw 27 m   -> "lowland" series (may contain NaN)
co2_mt   = frame[ST_MT  ].to_numpy()   # Jungfraujoch  -> "mountain" series
co2_izo  = frame[ST_IZO ].to_numpy()   # Izaña         -> subtropical background (Fig 2 only)
co2_nord = frame[ST_NORD].to_numpy()   # Station Nord  -> high-Arctic background (Fig 2 only)

times   = np.array(idx.to_pydatetime())
START   = idx[0].to_pydatetime()
hours   = np.arange(len(idx))
t_years = hours / (365.25 * 24)

for name, a in [("co2_low  (Cabauw 27 m)  ", co2_low), ("co2_mt   (Jungfraujoch) ", co2_mt),
                ("co2_izo  (Izaña)        ", co2_izo), ("co2_nord (Station Nord)  ", co2_nord)]:
    finite = np.isfinite(a)
    print(f"{name}: {finite.sum():>6d}/{a.size} hourly values "
          f"({100 * finite.mean():5.1f}% coverage), "
          f"range {np.nanmin(a):.1f}-{np.nanmax(a):.1f} ppm")


def fill_for_fft(a, index=idx):
    """NaN-free copy of an hourly series for FFT: time-interpolate interior gaps <= 7 days,
    fill longer gaps + the ends with the day-of-year climatology, then ffill/bfill any residue."""
    s = pd.Series(a, index=index).interpolate(method="time", limit=24 * 7, limit_area="inside")
    if s.isna().any():
        s = s.fillna(s.groupby(index.dayofyear).transform("mean"))
    return s.ffill().bfill().to_numpy()


co2_low_fft  = fill_for_fft(co2_low)
co2_mt_fft   = fill_for_fft(co2_mt)
co2_izo_fft  = fill_for_fft(co2_izo)
co2_nord_fft = fill_for_fft(co2_nord)
assert all(np.isfinite(a).all() for a in (co2_low_fft, co2_mt_fft, co2_izo_fft, co2_nord_fft))


def best_day_window(valid, n_days, prefer_months=(6, 7, 8), min_frac=0.8):
    """Day-aligned start index of the n_days window with the most valid hours, preferring
    windows centred in `prefer_months` once they clear `min_frac` coverage."""
    nh     = n_days * 24
    csum   = np.concatenate([[0], np.cumsum(valid.astype(int))])
    starts = np.arange(0, len(valid) - nh + 1, 24)
    win_ok = csum[starts + nh] - csum[starts]
    midmon = np.array([times[s + nh // 2].month for s in starts])
    pool   = np.where((win_ok >= min_frac * nh) & np.isin(midmon, prefer_months))[0]
    if pool.size == 0:
        pool = np.where(win_ok >= min_frac * nh)[0]
    if pool.size == 0:
        pool = np.arange(starts.size)
    return int(starts[pool[np.argmax(win_ok[pool])]])

## Figure 1 — the atmospheric CO₂ signal at three time scales (Cabauw 27 m)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 9.2), constrained_layout=True)

# ── A. Five years of daily means ─────────────────────────────────────────────
daily = np.nanmean(co2_low.reshape(-1, 24), axis=1)
days  = np.array([START + timedelta(days=int(d)) for d in range(len(daily))])
trend_smooth = pd.Series(daily).rolling(365, center=True, min_periods=1).mean().to_numpy()
half = 182
trend_plot = trend_smooth.copy()
trend_plot[:half]  = np.nan
trend_plot[-half:] = np.nan
N = len(daily)
# Robust y-limits: at the 27 m intake, strong nighttime build-up + winter pollution push the
# spikiest daily means past 500 ppm; clip the view to (1st .. 92nd)-percentile daily means so the
# trend and seasonal cycle stay legible (the few off-scale days clip at the top — that's the point).
lo, hi = (float(v) for v in np.nanpercentile(daily, [1, 92]))
_bbox = dict(boxstyle="round,pad=0.3", fc="white", ec="0.85", lw=0.5, alpha=0.9)

ax = axes[0]
ax.plot(days, daily, color=C_OBS, lw=0.6, alpha=0.6)
ax.plot(days, trend_plot, color=C_TREND, lw=2.6, label="1-year running mean (trend)")
ax.set_title("A. Five years of daily means — the trend and the seasonal cycle dominate")
ax.set_ylabel("CO$_2$  (ppm)")
ax.set_ylim(lo - 4, hi + 18)
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="lower right", frameon=True, framealpha=0.85, edgecolor="0.85")

# trend rate from a straight-line fit to the running-mean curve (ppm / yr)
m_valid = np.isfinite(trend_plot)
slope_ppm_yr = np.polyfit(np.arange(N)[m_valid], trend_plot[m_valid], 1)[0] * 365.0
ax.annotate(f"rising trend  ≈ {slope_ppm_yr:+.1f} ppm yr$^{{-1}}$\n(fossil-fuel accumulation)",
            xy=(days[int(0.45 * N)], trend_plot[int(0.45 * N)]),
            xytext=(days[int(0.50 * N)], hi + 15), fontsize=9.5, color=C_TREND, ha="left", va="top",
            bbox=_bbox, arrowprops=dict(arrowstyle="->", color=C_TREND, lw=1))

# winter maximum / summer minimum from one interior calendar year (2022 if available)
try:
    year_of  = np.array([d.year for d in days])
    month_of = np.array([d.month for d in days])
    ref_year = 2022 if (year_of == 2022).any() else int(np.bincount(year_of).argmax())
    yr_mask  = year_of == ref_year
    wi_pool  = np.where(yr_mask & np.isin(month_of, [1, 2, 12]))[0]
    su_pool  = np.where(yr_mask & np.isin(month_of, [6, 7, 8]))[0]
    winter_idx = int(wi_pool[np.nanargmax(daily[wi_pool])])
    summer_idx = int(su_pool[np.nanargmin(daily[su_pool])])
    ax.annotate("winter maximum\n(low photosynthesis)",
                xy=(days[winter_idx], min(float(daily[winter_idx]), hi + 7)),
                xytext=(days[int(0.04 * N)], hi + 15),
                fontsize=9, color=C_SEASONAL, ha="left", va="top", bbox=_bbox,
                arrowprops=dict(arrowstyle="->", color=C_SEASONAL, lw=0.9))
    ax.annotate("summer minimum\n(strong photosynthesis)",
                xy=(days[summer_idx], float(daily[summer_idx])),
                xytext=(days[min(summer_idx + int(0.05 * N), N - 1)], lo + 3.5),
                fontsize=9, color=C_SEASONAL, ha="left", va="center", bbox=_bbox,
                arrowprops=dict(arrowstyle="->", color=C_SEASONAL, lw=0.9))
except (ValueError, IndexError):
    pass   # not enough data in the reference year to place the seasonal callouts

# ── B. One month, hourly ─────────────────────────────────────────────────────
i0 = best_day_window(np.isfinite(co2_low), 30); i1 = i0 + 24 * 30
ax = axes[1]
ax.plot(times[i0:i1], co2_low[i0:i1], color=C_OBS, lw=0.8, alpha=0.85)
sm24 = pd.Series(co2_low).rolling(24, center=True, min_periods=6).mean().to_numpy()
ax.plot(times[i0:i1], sm24[i0:i1], color=C_SYNOPTIC, lw=2.4,
        label="24-hour running mean (synoptic envelope)")
ax.set_title("B. One month, hourly — weather systems sweep clean / polluted air past the tower")
ax.set_ylabel("CO$_2$  (ppm)")
ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.legend(loc="upper left")

# ── C. Five days, hourly ─────────────────────────────────────────────────────
i0 = best_day_window(np.isfinite(co2_low), 5); i1 = i0 + 24 * 5
ax = axes[2]
ax.plot(times[i0:i1], co2_low[i0:i1], color=C_OBS, lw=1.4, marker="o", ms=3)
ax.set_title("C. Five days, hourly — the diurnal cycle: nighttime build-up under a shallow "
             "boundary layer, daytime drawdown by photosynthesis")
ax.set_ylabel("CO$_2$  (ppm)")
ax.set_xlabel("Date")
ax.xaxis.set_major_locator(mdates.DayLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
for d in range(6):
    ns = times[i0] + timedelta(days=d, hours=18)
    ne = times[i0] + timedelta(days=d + 1, hours=6)
    ax.axvspan(ns, ne, color=C_TREND, alpha=0.06, lw=0)
ax.text(0.012, 0.96, "shaded = night", transform=ax.transAxes, fontsize=9, color=C_TREND, va="top")

fig.suptitle("The atmospheric CO$_2$ signal at one ICOS station (Cabauw, 27 m), viewed at three time scales",
             fontsize=13.5, fontweight="bold")
fig.savefig(OUTDIR / "fig1_three_scales.png", bbox_inches="tight")
plt.show()

## Figure 2 — Fourier power spectrum: Cabauw 27 m, Jungfraujoch, Izaña, Station Nord

Four stations, four "noise climates": the busy lowland tall tower (Cabauw 27 m) carries by far
the most variance at the synoptic and diurnal scales; the alpine (Jungfraujoch) and the two
clean global-background sites (Izaña — subtropical free troposphere; Station Nord — high Arctic)
flatten progressively until the annual cycle is essentially all that's left. The spectrum uses
the gap-filled hourly series; if the real PSD clips the y-limits, widen `ymin` / `ymax` below.

In [ ]:
def fft_psd(signal, dt_h=1.0):
    n = len(signal)
    x = signal - np.mean(signal)
    coef = np.polyfit(np.arange(n), x, 1)
    x = x - (coef[0] * np.arange(n) + coef[1])
    w = np.hanning(n)
    x = x * w
    norm = (w ** 2).sum()
    X = np.fft.rfft(x)
    f_per_h = np.fft.rfftfreq(n, d=dt_h)
    P = (np.abs(X) ** 2) / (norm * (1.0 / dt_h)) * 2
    f_per_day = f_per_h * 24
    P_day = P / 24
    mask = f_per_day > 0
    f_pos = f_per_day[mask]; P_pos = P_day[mask]
    log_edges = np.linspace(np.log10(f_pos.min()), np.log10(f_pos.max()), 350)
    edges = 10 ** log_edges
    P_binned, _ = np.histogram(f_pos, bins=edges, weights=P_pos)
    counts, _ = np.histogram(f_pos, bins=edges)
    centers = np.sqrt(edges[:-1] * edges[1:])
    valid = counts > 0
    return centers[valid], (P_binned[valid] / counts[valid])


fig, ax = plt.subplots(figsize=(11, 6.4), constrained_layout=True)

bands = [
    (1 / (2 * 365), 1 / 120, C_TREND,    "trend & multi-year"),
    (1 / 120,       1 / 30,  C_SEASONAL, "seasonal biosphere"),
    (1 / 30,        1 / 2,   C_SYNOPTIC, "synoptic weather"),
    (1 / 2,         6,       C_DIURNAL,  "diurnal & semi-diurnal"),
]
ymin, ymax = 1e-3, 1e5
for f0, f1, col, _ in bands:
    ax.axvspan(f0, f1, color=col, alpha=0.10, lw=0)

# Four stations, ordered noisiest -> cleanest; the busiest spectrum is drawn first/under.
spectra = [
    (co2_low_fft,  "Cabauw 27 m (NL lowland)",       ICOS_BLUE, 1.4),
    (co2_mt_fft,   "Jungfraujoch (CH alpine)",       "#444444", 1.2),
    (co2_izo_fft,  "Izaña (ES subtropical)",         "#c1440e", 1.2),
    (co2_nord_fft, "Station Nord (Greenland Arctic)", "#7b3294", 1.2),
]
for arr, lbl, col, lw in spectra:
    f, P = fft_psd(arr)
    ax.loglog(f, P, color=col, lw=lw, label=lbl, alpha=0.9)

key_freqs = [
    (1 / 365.25, "1 / year", C_SEASONAL),
    (1 / 7,      "1 / week", C_ANTHRO),
    (1.0,        "1 / day",  C_DIURNAL),
    (2.0,        "2 / day",  C_DIURNAL),
]
for f0, lbl, col in key_freqs:
    ax.axvline(f0, color=col, lw=1.0, ls="--", alpha=0.55)
    ax.text(f0 * 1.08, ymax * 0.45, lbl, color=col, fontsize=10, ha="left", va="top", fontweight="semibold")

for f0, f1, col, lbl in bands:
    ax.text(np.sqrt(f0 * f1), ymin * 2.5, lbl, ha="center", va="bottom", fontsize=9.5,
            color=col, fontweight="semibold", alpha=0.9)

ax.set_xlim(1 / (2 * 365), 6)
ax.set_ylim(ymin, ymax)
ax.set_xlabel("Frequency  (cycles / day)")
ax.set_ylabel("Power spectral density  (ppm$^2$ · day)")
ax.set_title("Fourier power spectrum: real ICOS CO$_2$ at four stations, decomposed by frequency", pad=22)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=4, fontsize=9,
          frameon=False, handlelength=1.6, columnspacing=1.8)

secax = ax.secondary_xaxis("top",
    functions=(lambda f: 1 / np.where(f == 0, np.nan, f),
               lambda T: 1 / np.where(T == 0, np.nan, T)))
secax.set_xticks([365, 30, 7, 1, 0.5])
secax.set_xticklabels(["1 yr", "1 mo", "1 wk", "1 d", "12 h"])
secax.set_xlabel("Period")

fig.savefig(OUTDIR / "fig2_power_spectrum.png", bbox_inches="tight")
plt.show()

## Figure 3 — sources & sinks (schematic), two-station comparison, time–space map

The original panel **(a)** plotted a *synthetic* component decomposition (trend / seasonal /
diurnal / synoptic / anthropogenic plumes) — those components don't exist in observed data, so
that panel is omitted. The remaining panels are renumbered **a / b / c**: panels **a** (monthly
sources & sinks) and **c** (the time–space process map) stay schematic / illustrative, while
panel **b** (two stations, two stories) now uses the real Cabauw 27 m + Jungfraujoch hourly series.

In [ ]:
fig = plt.figure(figsize=(11.5, 7.4), constrained_layout=True)
gs  = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.2])

# ── a. Sources & sinks — illustrative monthly fluxes (schematic, not data) ───
ax = fig.add_subplot(gs[0, 0])
months = np.arange(1, 13)
fossil = np.full(12, 10.0)
biosph = np.array([3, 4, 1, -3, -7, -10, -9, -6, -2, 2, 4, 4])
ocean  = np.full(12, -2.5)
ax.bar(months - 0.24, fossil, width=0.22, color=C_ANTHRO,   label="Fossil fuels (source)")
ax.bar(months,        biosph, width=0.22, color=C_SEASONAL, label="Biosphere (source / sink)")
ax.bar(months + 0.24, ocean,  width=0.22, color=ICOS_BLUE,  label="Ocean (sink)")
ax.axhline(0, color="#444", lw=0.8)
ax.set_xticks(months)
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_ylabel("Flux  (illustrative units)")
ax.set_ylim(-13, 14)
ax.set_title("a. Sources and sinks differ in sign and seasonality")
ax.legend(fontsize=8.5, loc="upper left", ncol=1)

# ── b. Two stations, two stories — real hourly CO2, 14 days ──────────────────
i0 = best_day_window(np.isfinite(co2_low) & np.isfinite(co2_mt), 14)
i1 = i0 + 24 * 14
ax = fig.add_subplot(gs[0, 1])
ax.plot(times[i0:i1], co2_low[i0:i1], color=ICOS_BLUE, lw=0.9, label="Cabauw 27 m (lowland)")
ax.plot(times[i0:i1], co2_mt[i0:i1],  color="#555",   lw=1.4, label="Jungfraujoch (mountain)")
ax.set_title(f"b. Two stations, two stories (14 days from {times[i0].strftime('%d %b %Y')})")
ax.set_ylabel("CO$_2$  (ppm)")
ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.legend(fontsize=9, loc="upper right")

# ── c. Where each process lives in time–space (schematic) ────────────────────
ax = fig.add_subplot(gs[1, :])
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(0.5 / 24, 30 * 365)
ax.set_ylim(0.3, 30000)
processes = [
    ("Traffic rush hour",                   0.5 / 24, 12 / 24,  0.5,  20,    C_ANTHRO),
    ("Diurnal biosphere\n+ boundary layer", 6 / 24,   36 / 24,  1,    300,   C_DIURNAL),
    ("Weekly anthropogenic\ncycle",         5,        10,       20,   200,   C_ANTHRO),
    ("Synoptic weather",                    2,        14,       300,  5000,  C_SYNOPTIC),
    ("Seasonal biosphere",                  60,       365,      300,  3000,  C_SEASONAL),
    ("Ocean uptake",                        180,      20 * 365, 4000, 25000, ICOS_BLUE),
    ("Fossil-fuel trend",                   365,      20 * 365, 30,   3000,  C_TREND),
]
for lbl, t0, t1, s0, s1, col in processes:
    ax.fill_between([t0, t1], [s0, s0], [s1, s1], color=col, alpha=0.22, edgecolor=col, linewidth=1.3)
    ax.text(np.sqrt(t0 * t1), np.sqrt(s0 * s1), lbl, ha="center", va="center", fontsize=9,
            color=col, fontweight="semibold")
ax.set_xticks([1 / 24, 1, 7, 30, 365, 365 * 10])
ax.set_xticklabels(["1 h", "1 d", "1 wk", "1 mo", "1 yr", "10 yr"])
ax.set_yticks([1, 10, 100, 1000, 10000])
ax.set_yticklabels(["1 km", "10 km", "100 km", "1000 km", "10000 km"])
ax.set_xlabel("Characteristic time scale")
ax.set_ylabel("Characteristic spatial scale")
ax.set_title("c. Where each process lives in time–space — what a single station sees and what a network is needed for")
ax.grid(True, which="both", alpha=0.25)

fig.suptitle("Sources, sinks, and what each leaves behind in the atmospheric CO$_2$ record",
             fontsize=13.5, fontweight="bold")
fig.savefig(OUTDIR / "fig3_sources_sinks.png", bbox_inches="tight")
plt.show()

In [ ]:
try:
    from IPython.display import Image, display
    _can_display = True
except ImportError:
    _can_display = False

print("Done — figures written to", OUTDIR.resolve())
for p in sorted(OUTDIR.glob("fig*.png")):
    print("  ", p.name)
    if _can_display:
        display(Image(filename=str(p)))